# Stage C2 — Composite Loss Assembly

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Sec. 3.4 (Eq. 3.16–3.20).

C1 built and unit-tested six penalty *formulas*. This notebook turns
them into one trainable objective: variance-weighted data loss
(Eq. 3.17), the six physics terms combined with C1's adaptive weight
$\lambda_j(x)$ (Eq. 3.18), weight-decay regularization (Eq. 3.19),
weight calibration so no term dominates, and the sigmoid schedule
(Eq. 3.20) that introduces physics gradually.

**Notation note, easy to conflate:** Eq. 3.16 has *two* different
weights per constraint — $\lambda_j(x)$ (C1's density/validity
modulation, varies **by collocation point**, fixed once computed) and
$w_j$ (Eq. 3.16's outer scalar, varies **by epoch** via the Eq. 3.20
schedule, same value for every point at a given epoch). $\lambda_j(x)$
is already folded into $\mathcal{L}_{\text{physics},j}$ itself
(Eq. 3.18); $w_j(t)$ multiplies that whole term from outside.

**Input:** `data/masters_data.xlsx`, `outputs/C1_collocation_points.csv`,
`outputs/C1_constraint_config.json`, `outputs/B1_selected_architecture.json`
**Output:** every figure and result table is saved to `outputs/html/` as
`C2_<section>[_qualifier].html`; `outputs/C2_loss_config.json` (read by
C3/C4) holds the calibrated $w_j^{\text{final}}$, the calibration rule
and the schedule parameters.

**Changes from the first version of this notebook:** the constraint
functions are C1's, copied verbatim and checked for equality (Section 3);
the calibration model is the B1/B2/C1 model with the η head; the weight
calibration no longer divides by a penalty that can be ≈ 0 and no longer
cancels C1's validity function (Section 5); the schedule parameters are
fixed here and documented as such (Section 6).

## Setup

In [ ]:
import json
import numpy as np
import polars as pl
import plotly.graph_objects as go
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("polars    ", pl.__version__)
import plotly
print("plotly    ", plotly.__version__)
print("tensorflow", tf.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
SEED = 42
RAW_PATH


## Color palette and output naming (shared across the whole pipeline)

`SPLIT_COLORS` (semantic role: train / validation / test / reference
value / alert / neutral) and `VARIABLE_COLORS` (identity of each of the
4 inputs and 5 outputs) are identical in every A/B/C notebook, so the
same element always has the same color in any chart of the pipeline.

Every figure or result table generated below is also saved to
`outputs/html/`, named `C2_<section>[_qualifier].html` — the number
matches the corresponding section header, so the order in which each
output was produced can be read from the file name alone.

In [ ]:
SPLIT_COLORS = {
    "train": "#B7C9DA",
    "validation": "#2B6EFF",
    "test": "#571D99",
    "reference": "#343A40",   # value transcribed from the dissertation text
    "alert": "#E85D04",       # outlier / out of range / anomaly
    "neutral": "#B0AFA8",     # grid lines / neutral reference
}
VARIABLE_COLORS = {
    "SOI": "#073b3a", "lambda": "#0b6e4f", "sub_rate": "#08a045", "P_rail": "#6bbf59",
    "NOx": "#c7adff", "PM": "#916dd5", "eta": "#7151a9", "HC": "#573d7f", "CO2": "#46325d",
}

HTML_DIR = OUT_DIR / "html"
HTML_DIR.mkdir(parents=True, exist_ok=True)


def flagged_table_html(df, title, out_path, flag_col=None, is_flagged=lambda v: False, ref_cols=()):
    """Result table -> Plotly go.Table -> HTML.
    Columns listed in ref_cols get the 'reference' tone in the header
    (values transcribed from the dissertation text). Cells in flag_col
    get the 'alert' tone wherever is_flagged(value) is True."""
    cols = list(df.columns)
    n = df.shape[0]
    header_fill = [SPLIT_COLORS["reference"] if c in ref_cols else "#F1F3F5" for c in cols]
    header_font = ["white" if c in ref_cols else "black" for c in cols]
    cell_fill = []
    for c in cols:
        if c == flag_col:
            cell_fill.append([SPLIT_COLORS["alert"] if is_flagged(v) else "white"
                               for v in df[c].to_list()])
        else:
            cell_fill.append(["white"] * n)
    fig = go.Figure(data=[go.Table(
        header=dict(values=cols, fill_color=header_fill,
                     font=dict(color=header_font), align="left"),
        cells=dict(values=[df[c].to_list() for c in cols],
                    fill_color=cell_fill, align="left"),
    )])
    fig.update_layout(title=title, margin=dict(t=40, l=10, r=10, b=10))
    fig.write_html(str(out_path), include_plotlyjs="inline")
    return fig


def simple_table_html(df, title, out_path):
    return flagged_table_html(df, title, out_path)

# Constraint identity (not covered by the two palettes above): each constraint takes the
# color of the output it constrains, and the dash pattern tells same-output constraints apart.
CONSTRAINT_STYLE = {
    "NOx-SOI": dict(color=VARIABLE_COLORS["NOx"], dash="solid"),
    "PM-lambda": dict(color=VARIABLE_COLORS["PM"], dash="solid"),
    "HC-lambda": dict(color=VARIABLE_COLORS["HC"], dash="solid"),
    "NOx-PM": dict(color=VARIABLE_COLORS["PM"], dash="dash"),
    "eta-NOx": dict(color=VARIABLE_COLORS["eta"], dash="solid"),
    "non-negativity": dict(color=SPLIT_COLORS["reference"], dash="dot"),
}

## 1. Load data, architecture, and C1's collocation points

Reuses C1's 1000 LHS points and their precomputed $\rho(x)/\rho_{\max}$
and $v_{\eta\text{-NOx}}(x)$ instead of regenerating them — same
collocation set throughout C1-C4 matters more than re-deriving it
matters little.

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS
N_IN, N_OUT = len(INPUT_COLS), len(OUTPUT_COLS)
SOI_IDX, LAMBDA_IDX, SUBRATE_IDX, PRAIL_IDX = [INPUT_COLS.index(c) for c in INPUT_COLS]
HC_IDX, NOX_IDX, CO2_IDX, PM_IDX, ETA_IDX = [OUTPUT_COLS.index(c) for c in OUTPUT_COLS]
EMISSION_IDXS = [HC_IDX, NOX_IDX, CO2_IDX, PM_IDX]

df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]
medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}
deviation = np.column_stack([np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]

def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)

ofat_block = smooth_isolated_labels(raw_block)
extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    pos = np.array([0.5]) if m == 1 else np.empty(m)
    if m > 1:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2
rng_np = np.random.default_rng(SEED)
jitter = rng_np.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))
split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"
df = df.with_columns(pl.Series("split", split))
train_df = df.filter(pl.col("split") == "train")
train_min = {c: train_df[c].min() for c in ALL_COLS}
train_max = {c: train_df[c].max() for c in ALL_COLS}
df = df.with_columns([
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
])
X_train = df.filter(pl.col("split") == "train").select([f"{c}_norm" for c in INPUT_COLS]).to_numpy().astype(np.float32)
Y_train = df.filter(pl.col("split") == "train").select([f"{c}_norm" for c in OUTPUT_COLS]).to_numpy().astype(np.float32)

# ---- C1 outputs: collocation points + constraint configuration ----
colloc_path = OUT_DIR / "C1_collocation_points.csv"
config_path = OUT_DIR / "C1_constraint_config.json"
for p in (colloc_path, config_path):
    if not p.exists():
        raise FileNotFoundError(f"{p} not found -- run C1 (including its save cell) before C2.")
colloc_df = pl.read_csv(colloc_path)
X_colloc = colloc_df.select(INPUT_COLS).to_numpy().astype(np.float32)
rho_colloc = colloc_df["density_ratio"].to_numpy()
validity_eff_nox = colloc_df["validity_eff_nox"].to_numpy()
with open(config_path) as f:
    C1_CONFIG = json.load(f)

# the non-negativity floors must be the ones C1 computed from this same split
EMISSIONS = [OUTPUT_COLS[i] for i in EMISSION_IDXS]
floors_here = [float(-train_min[c] / (train_max[c] - train_min[c])) for c in EMISSIONS]
floors_c1 = [C1_CONFIG["nonneg_floor_norm"][c] for c in EMISSIONS]
if not np.allclose(floors_here, floors_c1):
    raise ValueError("non-negativity floors differ from C1 -- the data or split changed; rerun C1.")

# ---- B1 architecture ----
selection_path = OUT_DIR / "B1_selected_architecture.json"
if not selection_path.exists():
    raise FileNotFoundError(f"{selection_path} not found -- run B1 (including its save cell) before C2.")
with open(selection_path) as f:
    selection = json.load(f)
HIDDEN_UNITS = tuple(selection["hidden_units"])
print(f"Loaded {X_colloc.shape[0]} collocation points from C1; architecture {selection['architecture_id']} {HIDDEN_UNITS}")
print(f"C1 config: NOx-SOI direction = {C1_CONFIG['nox_soi_direction']}, validity rule = {C1_CONFIG['validity_eff_nox_rule']}")

## 2. Data loss (Eq. 3.17) — variance-weighted MSE

$$
\mathcal{L}_{\text{data}} = \frac{1}{N}\sum_{i=1}^{N}\sum_{k=1}^{5} \frac{1}{\sigma_k^2}\left(y^{\text{pred}}_{i,k} - y^{\text{exp}}_{i,k}\right)^2
$$

$\sigma_k^2$ from the **training data**, normalized outputs, held
fixed for the rest of the pipeline — not recomputed per batch.

In [ ]:
sigma2 = Y_train.var(axis=0, ddof=1)
sigma2 = np.maximum(sigma2, 1e-8)  # guard against a near-constant output

def data_loss(predict_fn, X, Y, sigma2=sigma2):
    X_t = tf.convert_to_tensor(X, dtype=tf.float32)
    Y_t = tf.convert_to_tensor(Y, dtype=tf.float32)
    pred = predict_fn(X_t)
    sq_err = tf.square(pred - Y_t) / tf.constant(sigma2, dtype=tf.float32)
    return tf.reduce_mean(tf.reduce_sum(sq_err, axis=1))

variances = pl.DataFrame({"output": OUTPUT_COLS, "sigma2_train_norm": np.round(sigma2, 6),
                          "weight_1_over_sigma2": np.round(1 / sigma2, 3)})
simple_table_html(variances, "C2 -- Per-output variance of the normalized train targets (Eq. 3.17 weights)",
                   HTML_DIR / "C2_02_data_loss_variances.html")
variances

## 3. Physics loss (Eq. 3.18) — C1's formulas, now weighted by $\lambda_j(x)$

$$
\mathcal{L}_{\text{physics},j} = \frac{1}{N_c}\sum_{c=1}^{N_c} \lambda_j(x_c)\, p_j(x_c),
\qquad
p_j(x) = \max\!\big(0,\ g_j(x)\big)^2,
\qquad
\lambda_j(x) = \frac{\rho(x)}{\rho_{\max}}\, v_j(x)
$$

where $g_j$ is the quantity whose positive part is a violation:
$\pm\partial\text{NOx}/\partial\text{SOI}$ (Eq. 3.9, sign set by C1's
direction), $\partial\text{PM}/\partial\lambda$ (Eq. 3.10),
$-\partial^2\text{HC}/\partial\lambda^2$ (Eq. 3.11), and the products of
SOI-derivatives for the two trade-offs (Eq. 3.12–3.13). Non-negativity
(Eq. 3.14) has no $\lambda_j(x)$ term, so it stays a plain mean of the
per-point sum over the four emissions, with C1's physical-zero floors.

The first cell copies **C1's six constraint functions verbatim**. The
second cell defines the per-point versions C2 needs (the pointwise weight
has to be applied before averaging), and the third checks that, without
the weight, each per-point version averages to exactly C1's value on the
same network and points — so C1's unit tests carry over to what C2–C4
actually train against.

In [ ]:
# ---- C1's constraint settings (from C1_constraint_config.json) ----
NOX_SOI_DIRECTION = C1_CONFIG["nox_soi_direction"]
PM_LAMBDA_DIRECTION = C1_CONFIG["pm_lambda_direction"]
NONNEG_FLOOR_NORM = floors_c1

# ---- C1's constraint functions, copied verbatim ----
def monotonic_constraint(predict_fn, x, out_idx, in_idx, direction):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        target = y[:, out_idx]
    grad = tape.gradient(target, x_t)
    d = grad[:, in_idx]
    violation = d if direction == "decreasing" else -d    # the derivative sign that is NOT allowed
    return tf.reduce_mean(tf.square(tf.maximum(0.0, violation)))


def nox_soi_constraint(predict_fn, x, direction=None):
    return monotonic_constraint(predict_fn, x, NOX_IDX, SOI_IDX, direction or NOX_SOI_DIRECTION)


def pm_lambda_constraint(predict_fn, x):
    return monotonic_constraint(predict_fn, x, PM_IDX, LAMBDA_IDX, PM_LAMBDA_DIRECTION)


def convexity_constraint(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape2:
        tape2.watch(x_t)
        with tf.GradientTape() as tape1:
            tape1.watch(x_t)
            y = predict_fn(x_t)
            target = y[:, out_idx]
        grad1 = tape1.gradient(target, x_t)
        d_first = grad1[:, in_idx]
    grad2 = tape2.gradient(d_first, x_t)
    d_second = grad2[:, in_idx]
    return tf.reduce_mean(tf.square(tf.maximum(0.0, -d_second)))


def hc_lambda_constraint(predict_fn, x):
    return convexity_constraint(predict_fn, x, HC_IDX, LAMBDA_IDX)


def tradeoff_constraint(predict_fn, x, out_idx_a, out_idx_b, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        a = y[:, out_idx_a]
        b = y[:, out_idx_b]
    grad_a = tape.gradient(a, x_t)[:, in_idx]
    grad_b = tape.gradient(b, x_t)[:, in_idx]
    del tape
    return tf.reduce_mean(tf.square(tf.maximum(0.0, grad_a * grad_b)))


def nox_pm_tradeoff_constraint(predict_fn, x):
    return tradeoff_constraint(predict_fn, x, NOX_IDX, PM_IDX, SOI_IDX)


def eff_nox_tradeoff_constraint(predict_fn, x):
    return tradeoff_constraint(predict_fn, x, ETA_IDX, NOX_IDX, SOI_IDX)


def nonneg_constraint(predict_fn, x, out_idxs=EMISSION_IDXS, floors=NONNEG_FLOOR_NORM):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    y = predict_fn(x_t)
    emissions = tf.gather(y, out_idxs, axis=1)
    floor_t = tf.constant(floors, dtype=tf.float32)          # normalized value of physical zero
    # Eq. 3.14: sum over the four emissions, mean over points
    return tf.reduce_mean(tf.reduce_sum(tf.square(tf.maximum(0.0, floor_t - emissions)), axis=1))

In [ ]:
def g_monotonic(predict_fn, x, out_idx, in_idx, direction):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_t)
        target = predict_fn(x_t)[:, out_idx]
    d = tape.gradient(target, x_t)[:, in_idx]
    return d if direction == "decreasing" else -d

def g_convexity(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape2:
        tape2.watch(x_t)
        with tf.GradientTape() as tape1:
            tape1.watch(x_t)
            target = predict_fn(x_t)[:, out_idx]
        d_first = tape1.gradient(target, x_t)[:, in_idx]
    d_second = tape2.gradient(d_first, x_t)[:, in_idx]
    return -d_second

def g_tradeoff(predict_fn, x, out_idx_a, out_idx_b, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        a, b = y[:, out_idx_a], y[:, out_idx_b]
    grad_a = tape.gradient(a, x_t)[:, in_idx]
    grad_b = tape.gradient(b, x_t)[:, in_idx]
    del tape
    return grad_a * grad_b

def g_nonneg(predict_fn, x, out_idxs=EMISSION_IDXS, floors=NONNEG_FLOOR_NORM):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    emissions = tf.gather(predict_fn(x_t), out_idxs, axis=1)
    return tf.constant(floors, dtype=tf.float32) - emissions        # [N, 4]

G_FUNCTIONS = {
    "NOx-SOI": lambda f, x: g_monotonic(f, x, NOX_IDX, SOI_IDX, NOX_SOI_DIRECTION),
    "PM-lambda": lambda f, x: g_monotonic(f, x, PM_IDX, LAMBDA_IDX, PM_LAMBDA_DIRECTION),
    "HC-lambda": lambda f, x: g_convexity(f, x, HC_IDX, LAMBDA_IDX),
    "NOx-PM": lambda f, x: g_tradeoff(f, x, NOX_IDX, PM_IDX, SOI_IDX),
    "eta-NOx": lambda f, x: g_tradeoff(f, x, ETA_IDX, NOX_IDX, SOI_IDX),
    "non-negativity": lambda f, x: g_nonneg(f, x),
}
CONSTRAINT_NAMES = list(G_FUNCTIONS)
GRADIENT_CONSTRAINTS = CONSTRAINT_NAMES[:5]          # these carry lambda_j(x); non-negativity does not

def per_point_penalty(g):
    p = tf.square(tf.maximum(0.0, g))
    return tf.reduce_sum(p, axis=1) if len(g.shape) > 1 else p

def per_point_magnitude(g):
    m = tf.square(g)
    return tf.reduce_sum(m, axis=1) if len(g.shape) > 1 else m

VALIDITY = {name: np.ones(len(X_colloc)) for name in GRADIENT_CONSTRAINTS}
VALIDITY["eta-NOx"] = validity_eff_nox
LAMBDA_X = {name: rho_colloc * VALIDITY[name] for name in GRADIENT_CONSTRAINTS}   # lambda_j^0 = 1 here

def physics_loss(name, predict_fn, x=X_colloc):
    p = per_point_penalty(G_FUNCTIONS[name](predict_fn, x))
    if name in LAMBDA_X:
        return tf.reduce_mean(tf.constant(LAMBDA_X[name], dtype=tf.float32) * p)
    return tf.reduce_mean(p)

In [ ]:
# Calibration network (same builder as B1/B2/C1), reused in Section 5, and the C2 == C1 check
assert OUTPUT_COLS[-1] == "eta", "the eta head is appended last; OUTPUT_COLS must end with eta"

# eta as a physical fraction (A1 Section 7: stored as a fraction) and its train-only range (A3 Section 9)
to_fraction = (lambda v: v / 100) if df["eta"].max() > 1 else (lambda v: v)
ETA_MIN, ETA_MAX = to_fraction(train_min["eta"]), to_fraction(train_max["eta"])
ETA_MEAN_TRAIN = float(np.mean(to_fraction(df.filter(pl.col("split") == "train")["eta"].to_numpy())))
ETA_BIAS_INIT = float(np.log(ETA_MEAN_TRAIN / (1 - ETA_MEAN_TRAIN)))    # logit(mean train eta)


def build_model(hidden_units, seed, input_dim=N_IN, output_dim=N_OUT):
    tf.random.set_seed(seed)
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    for units in hidden_units:
        x = layers.Dense(units, activation="tanh")(x)
    emissions = layers.Dense(output_dim - 1, activation="linear", name="emissions")(x)   # HC, NOx, CO2, PM
    eta_frac = layers.Dense(1, activation="sigmoid", name="eta_fraction",
                            bias_initializer=keras.initializers.Constant(ETA_BIAS_INIT))(x)
    eta_norm = layers.Rescaling(scale=1.0 / (ETA_MAX - ETA_MIN),
                                offset=-ETA_MIN / (ETA_MAX - ETA_MIN), name="eta_rescaled")(eta_frac)
    outputs = layers.Concatenate(name="outputs")([emissions, eta_norm])
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="mse")
    return model


def predict_np(model, X):
    """Forward pass as a NumPy array, without model.predict().

    model.predict() builds a new tf.function for every freshly built model;
    in a loop of 240 fits that triggers TensorFlow's "tf.function retracing"
    warning and is slower than a direct call on arrays this small. A direct
    call with training=False gives the same predictions (no dropout or batch
    normalization in these models)."""
    return np.asarray(model(np.asarray(X, dtype="float32"), training=False))


calib_model = build_model(HIDDEN_UNITS, seed=SEED)
predict_fn = lambda x: calib_model(x, training=False)

C1_FUNCTION_OF = {"NOx-SOI": nox_soi_constraint, "PM-lambda": pm_lambda_constraint,
                  "HC-lambda": hc_lambda_constraint, "NOx-PM": nox_pm_tradeoff_constraint,
                  "eta-NOx": eff_nox_tradeoff_constraint, "non-negativity": nonneg_constraint}
eq_rows = []
for name in CONSTRAINT_NAMES:
    c1_value = float(C1_FUNCTION_OF[name](predict_fn, X_colloc))
    c2_value = float(tf.reduce_mean(per_point_penalty(G_FUNCTIONS[name](predict_fn, X_colloc))))
    eq_rows.append({"constraint": name, "C1_function": float(f"{c1_value:.8g}"),
                    "C2_per_point_mean": float(f"{c2_value:.8g}"),
                    "equal": bool(np.isclose(c1_value, c2_value, rtol=1e-5, atol=1e-10))})
c1_c2_equality = pl.DataFrame(eq_rows)
flagged_table_html(c1_c2_equality, "C2 -- Per-point penalties (unweighted mean) vs. C1's functions, same network",
                    HTML_DIR / "C2_03_c1_c2_equality.html", flag_col="equal", is_flagged=lambda v: not v)
assert all(c1_c2_equality["equal"].to_list()), "C2's per-point penalties do not reproduce C1's functions"
c1_c2_equality

## 4. Regularization (Eq. 3.19) — weight decay, biases excluded

$$
\mathcal{L}_{\text{reg}} = \frac{1}{P}\sum_{l=1}^{L}\sum_{i,j}\left(w^{(l)}_{ij}\right)^2
$$

$P$ is the model's **total** parameter count (Sec. 3.4.1.3 — same
number B1 computes via `count_params`), even though the sum itself only
touches kernels (2D weight matrices), not biases (1D). The `Rescaling`
layer of the η head has no weights, so it adds nothing here.

In [ ]:
def regularization_loss(model):
    P = model.count_params()
    sq_sum = tf.add_n([tf.reduce_sum(tf.square(w)) for w in model.trainable_weights if len(w.shape) > 1])
    return sq_sum / P


## 5. Weight calibration — $w_j^{\text{final}}$

**The text's rule** (Sec. 3.4.2): evaluate $\mathcal{L}_{\text{data}}$ and
every $\mathcal{L}_{\text{physics},j}$ on an untrained network and set
$w_j^{\text{final}} = \mathcal{L}_{\text{data}} / \mathcal{L}_{\text{physics},j}$,
so physics matches data once the schedule reaches full strength.

**Two problems with applying it literally**, both visible in the table
below:

1. *Division by ≈ 0.* $p_j = \max(0, g_j)^2$ is exactly zero wherever the
   random network happens to satisfy the constraint. When that holds at
   (almost) every collocation point, $\mathcal{L}_{\text{physics},j}$ is
   ≈ 0 and $w_j^{\text{final}}$ explodes — the first violation during
   training then outweighs the data term by orders of magnitude.
2. *It cancels the validity function.* $\mathcal{L}_{\text{physics},j}$
   includes $\lambda_j(x) = (\rho/\rho_{\max})\,v_j(x)$. Dividing by it
   scales $w_j$ up by exactly as much as $v_j$ scaled the term down. For
   η–NOx, where C1's validity is ≈ 0 almost everywhere (Sec. 3.3.2.2),
   the calibration would switch the constraint back on at full strength.

**Rule implemented** (`CALIBRATION_MODE = "magnitude"`): calibrate on the
*natural scale* of each term — the mean of $g_j^2$ over the collocation
points, without the hinge and without $\lambda_j(x)$:

$$
w_j^{\text{final}} = \frac{\mathcal{L}_{\text{data}}^{\text{init}}}{\tfrac{1}{N_c}\sum_c g_j(x_c)^2}
$$

It is never ≈ 0 for a generic network, and it leaves $\lambda_j(x)$ with
its intended absolute meaning. Interpretation: at full schedule strength,
a constraint violated at every collocation point with full weight
($\lambda_j = 1$) contributes as much as the initial data loss; where
$v_j$ or the density is low, it contributes proportionally less. The
other two rules are computed alongside for comparison and can be
selected with the same setting. **Sec. 3.4.2 should be updated to
describe the rule actually used.**

In [ ]:
CALIBRATION_MODE = "magnitude"      # "magnitude" (implemented) | "text_literal" | "hinged_unweighted"
ILL_CONDITIONED_RATIO = 1e-6         # denominator below this fraction of L_data -> division by ~0

L_data_init = float(data_loss(predict_fn, X_train, Y_train))
L_reg_init = float(regularization_loss(calib_model))

calib_rows, w_candidates, L_physics_init = [], {}, {}
for name in CONSTRAINT_NAMES:
    g = G_FUNCTIONS[name](predict_fn, X_colloc)
    denom = {
        "text_literal": float(physics_loss(name, predict_fn)),                    # with lambda_j(x) and hinge
        "hinged_unweighted": float(tf.reduce_mean(per_point_penalty(g))),         # hinge, no lambda_j(x)
        "magnitude": float(tf.reduce_mean(per_point_magnitude(g))),               # no hinge, no lambda_j(x)
    }
    L_physics_init[name] = denom["text_literal"]
    w_candidates[name] = {m: L_data_init / max(d, 1e-12) for m, d in denom.items()}
    ill = {m: d < ILL_CONDITIONED_RATIO * L_data_init for m, d in denom.items()}
    calib_rows.append({
        "constraint": name,
        "L_physics_init (text)": float(f"{denom['text_literal']:.4g}"),
        "w_text_literal": float(f"{w_candidates[name]['text_literal']:.4g}"),
        "w_hinged_unweighted": float(f"{w_candidates[name]['hinged_unweighted']:.4g}"),
        "mean_g2 (magnitude)": float(f"{denom['magnitude']:.4g}"),
        "w_magnitude": float(f"{w_candidates[name]['magnitude']:.4g}"),
        "text_rule_ill_conditioned": bool(ill["text_literal"]),
        "used_rule": CALIBRATION_MODE,
    })
w_final = {name: w_candidates[name][CALIBRATION_MODE] for name in CONSTRAINT_NAMES}
if any(r["text_rule_ill_conditioned"] for r in calib_rows) and CALIBRATION_MODE == "text_literal":
    print("WARNING: the text-literal rule divides by ~0 for at least one constraint (see table)")

calibration = pl.DataFrame(calib_rows)
flagged_table_html(calibration,
                    f"C2 -- Weight calibration (L_data_init = {L_data_init:.5g}); alert = text rule divides by ~0",
                    HTML_DIR / "C2_05_calibration_table.html",
                    flag_col="text_rule_ill_conditioned", is_flagged=lambda v: v)
print(f"L_data (init) = {L_data_init:.5f}   L_reg (init) = {L_reg_init:.6f}   rule used: {CALIBRATION_MODE}")
calibration

## 6. Sigmoid weight schedule (Eq. 3.20)

$$
w_j(t) = \frac{w_j^{\text{final}}}{1 + \exp[-k(t-t_0)]}
$$

$k = 0.01$ and $t_0 = 750$ epochs are **fixed here** and used unchanged
by C3 and C4: Table 8 does not list them and C3 does not search them.
With these values the physics weight is ≈ 0.05 % of its final value at
epoch 0, half at epoch 750 and ≈ 99.9 % from epoch ≈ 1450. If $k$/$t_0$
should become hyperparameters, they need to be added to Table 8 and to
C3's search space. The table checks the mechanics: near 0 early,
$w_j^{\text{final}}/2$ at $t_0$, approaching $w_j^{\text{final}}$ late.

In [ ]:
K_SCHEDULE = 0.01
T0_SCHEDULE = 750

def scheduled_weight(t, w_j_final, k=K_SCHEDULE, t0=T0_SCHEDULE):
    return w_j_final / (1 + np.exp(-k * (t - t0)))

sched_rows = []
for name in w_final:
    sched_rows.append({"constraint": name, "w_final": float(f"{w_final[name]:.5g}"),
                       "w(0)": float(f"{scheduled_weight(0, w_final[name]):.5g}"),
                       "w(t0)": float(f"{scheduled_weight(T0_SCHEDULE, w_final[name]):.5g}"),
                       "w(2000)": float(f"{scheduled_weight(2000, w_final[name]):.5g}"),
                       "w(t0) == w_final/2": bool(np.isclose(scheduled_weight(T0_SCHEDULE, w_final[name]),
                                                             w_final[name] / 2))})
schedule_values = pl.DataFrame(sched_rows)
flagged_table_html(schedule_values, f"C2 -- Schedule check (k = {K_SCHEDULE}, t0 = {T0_SCHEDULE})",
                    HTML_DIR / "C2_06_schedule_values.html",
                    flag_col="w(t0) == w_final/2", is_flagged=lambda v: not v)
schedule_values

**The schedule itself**, all six constraints:

In [ ]:
t_range = np.linspace(0, 2000, 300)
fig = go.Figure()
for name, wf in w_final.items():
    st = CONSTRAINT_STYLE[name]
    fig.add_trace(go.Scatter(x=t_range, y=scheduled_weight(t_range, wf), mode="lines", name=name,
                             line=dict(color=st["color"], dash=st["dash"], width=2.5)))
fig.add_vline(x=T0_SCHEDULE, line_dash="dot", line_color=SPLIT_COLORS["neutral"], annotation_text="t0")
fig.update_layout(title=f"w_j(t): sigmoid ramp per constraint (calibration rule: {CALIBRATION_MODE})",
                   xaxis_title="epoch", yaxis=dict(title="w_j(t)", type="log"), width=850, height=450)
fig.show()
fig.write_html(str(HTML_DIR / "C2_06_schedule_curves.html"), include_plotlyjs="inline")

## 7. Composite loss (Eq. 3.16) — the function C4 actually trains against

$$
\mathcal{L}_{\text{total}}(t) = w_{\text{data}}\mathcal{L}_{\text{data}} + \sum_{j=1}^{M} w_j(t)\,\mathcal{L}_{\text{physics},j} + w_{\text{reg}}\mathcal{L}_{\text{reg}}
$$

$w_{\text{data}} = 1$. $w_{\text{reg}} = 10^{-5}$ is the fixed value of
Sec. 3.4.1, used **only for the smoke test and the illustration in this
notebook**: Table 8 lists the weight decay as a tuned hyperparameter, and
C3/C4 use the tuned value (the text should settle on one of the two
readings). `total_loss` can also return every component, which is how
the smoke-test table below is built.

In [ ]:
W_DATA = 1.0
W_REG = 1e-5     # illustration only -- C3/C4 pass the tuned weight decay (Table 8)

def total_loss(model, X, Y, epoch, w_final, w_reg=W_REG, training=True, return_components=False):
    predict_fn = lambda x: model(x, training=training)
    l_data = data_loss(predict_fn, X, Y)
    components = {"data": l_data}
    l_phys = 0.0
    for name in CONSTRAINT_NAMES:
        term = float(scheduled_weight(epoch, w_final[name])) * physics_loss(name, predict_fn)
        components[name] = term
        l_phys = l_phys + term
    l_reg = regularization_loss(model)
    components["reg"] = w_reg * l_reg
    total = W_DATA * l_data + l_phys + w_reg * l_reg
    return (total, components) if return_components else total

smoke_rows = []
for ep in [0, T0_SCHEDULE, 2000]:
    tot, comp = total_loss(calib_model, X_train, Y_train, ep, w_final, training=False, return_components=True)
    row = {"epoch": ep, "total": float(f"{float(tot):.6g}")}
    row.update({k: float(f"{float(v):.6g}") for k, v in comp.items()})
    row["finite"] = bool(np.isfinite(float(tot)))
    smoke_rows.append(row)
smoke = pl.DataFrame(smoke_rows)
flagged_table_html(smoke, "C2 -- total_loss smoke test on the untrained network, by component",
                    HTML_DIR / "C2_07_total_loss_smoke.html", flag_col="finite", is_flagged=lambda v: not v)
assert all(smoke["finite"].to_list())
smoke

## 8. Does physics vanish or dominate? (feedback: does the physics loss disappear or take over)

**Caveat up front:** this fixes $\mathcal{L}_{\text{data}}$ and each
$\mathcal{L}_{\text{physics},j}$ at their *initialization* values and
only varies $w_j(t)$ — it shows the schedule's intended shape, not a real
training trajectory (both loss values change once the network trains,
which only **C4** can show).

What it does show: early on the physics terms are negligible next to the
data term, so the network first fits the data; as the schedule ramps up,
each term grows toward $w_j^{\text{final}}\,\mathcal{L}_{\text{physics},j}$.
With the calibration rule of Section 5 that final value equals
$\mathcal{L}_{\text{data}}^{\text{init}}$ only for a constraint violated
everywhere at full weight; a constraint the network already satisfies in
most of the domain — or one whose validity is low, like η–NOx — stays
below the data term instead of being forced up to it.

In [ ]:
t_range2 = np.linspace(0, 2000, 100)
data_contribution = np.full_like(t_range2, W_DATA * L_data_init)
reg_contribution = np.full_like(t_range2, W_REG * L_reg_init)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_range2, y=data_contribution, mode="lines", name="data term (fixed L_data_init)",
                          line=dict(color=SPLIT_COLORS["train"], width=3)))
total_phys = np.zeros_like(t_range2)
for name in CONSTRAINT_NAMES:
    contrib = scheduled_weight(t_range2, w_final[name]) * L_physics_init[name]
    total_phys += contrib
    st = CONSTRAINT_STYLE[name]
    fig.add_trace(go.Scatter(x=t_range2, y=np.maximum(contrib, 1e-12), mode="lines", name=name,
                             line=dict(color=st["color"], dash=st["dash"], width=1.5)))
fig.add_trace(go.Scatter(x=t_range2, y=np.maximum(total_phys, 1e-12), mode="lines",
                          name="all physics terms (schedule x fixed L_physics_init)",
                          line=dict(color=SPLIT_COLORS["reference"], width=3)))
fig.add_trace(go.Scatter(x=t_range2, y=reg_contribution, mode="lines", name="reg term",
                          line=dict(color=SPLIT_COLORS["neutral"], width=1.5, dash="dot")))
fig.update_layout(title="Illustrative loss-component magnitudes under the schedule (not a real training run)",
                   xaxis_title="epoch", yaxis=dict(title="loss contribution (floored at 1e-12)", type="log"),
                   width=900, height=480)
fig.show()
fig.write_html(str(HTML_DIR / "C2_08_component_magnitudes.html"), include_plotlyjs="inline")

## Persist outputs

`C2_loss_config.json` is what C3 and C4 read: the calibrated weights for
the rule in use, the candidates of the other two rules, the schedule
parameters and the C1 settings the weights were calibrated with.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
config = {
    "w_data": W_DATA, "w_reg_illustration_only": W_REG,
    "calibration_mode": CALIBRATION_MODE,
    "w_final": w_final,
    "w_final_candidates": w_candidates,
    "k_schedule": K_SCHEDULE, "t0_schedule": T0_SCHEDULE,
    "L_data_init": L_data_init, "L_physics_init": L_physics_init, "L_reg_init": L_reg_init,
    "c1_config": C1_CONFIG,
    "architecture_id": selection["architecture_id"], "hidden_units": list(HIDDEN_UNITS),
}
with open(OUT_DIR / "C2_loss_config.json", "w") as f:
    json.dump(config, f, indent=2)
print(f"Saved to {OUT_DIR}")

## Next

**C3** searches Table 8's hyperparameter space (learning rate, weight
decay, dropout, activation, the $\lambda_j^0$ scalings, …) with the
physics terms built exactly as here; $k$/$t_0$ stay fixed at this
notebook's values (Section 6). **C4** trains the ensemble against
`total_loss` exactly as assembled here, with the tuned weight decay in
place of `W_REG`.